Importing Needed Libraries - Routing to Proper Device

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image

print("torch:", torch.__version__)
device = "cuda"  # Change device in the runtime settings - to GPU T4
print(torch.cuda.is_available())
print("device:", device)

def set_seed(seed: int = 42):                                                                       # Here for reproducability
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    # os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)                                                                                        # Here for reproducability
torch.backends.cudnn.benchmark = True

# malco was here
# sarah was here
# kat was here

print("hello")

torch: 2.11.0+cu128
True
device: cuda
hello


Importing From Kaggle API - Get Kaggle key first - Already Downloaded

In [2]:
!export KAGGLE_API_TOKEN=KGAT_6e0db17c19cd9ec6f55621b971c1014d

!mkdir -p ~/.kaggle && echo KGAT_6e0db17c19cd9ec6f55621b971c1014d > ~/.kaggle/access_token && chmod 666 ~/.kaggle/access_token

!kaggle competitions list

!export KAGGLE_API_TOKEN=KGAT_d77bc5e53c781f271763950cf13abd1c

!mkdir -p ~/.kaggle && echo KGAT_d77bc5e53c781f271763950cf13abd1c > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

!kaggle competitions list

ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                               2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                       2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection             2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3          

In [ ]:
# Code that kaggle said to run to get access to the competition files
import kagglehub

# Download latest version
path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

print("Path to competition files:", path)

 79%|███████▉  | 95.0M/120M [00:00<00:00, 121MB/s]

Data Pipeline - Dataset and DataLoader - IN PROGRESS

In [ ]:

batch_size = 128  # For now, can change to 64?
num_workers = 0                                                                 # Here for reproducability
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]
val_p = 0.15 # portion of training data


class UnlabeledDataset(Dataset):
  def __init__(self, root, transform=None):
      self.root   = root
      self.transform = transform
      self.images    = sorted(os.listdir(root))

  def __len__(self):
      return len(self.images)

  def __getitem__(self, idx):
      img_path = os.path.join(self.root, self.images[idx])
      image    = Image.open(img_path).convert('RGB')
      if self.transform:
          image = self.transform(image)
      return image, self.images[idx]

train_path = path + '/train'
test_path = path + '/test'

#data augmentation to expand training dataset
train_tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.Resize((256, 256)),
    v2.RandomResizedCrop(224, scale=(0.5, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.1),
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean, std),
    v2.RandomErasing(p=0.4, scale=(0.02, 0.15)),
])

test_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean, std)
])

train_set = datasets.ImageFolder(root=train_path, transform=train_tf)
val_set   = datasets.ImageFolder(root=train_path,  transform=test_tf)
test_set   = UnlabeledDataset(root=test_path,  transform=test_tf)

train_size = int((1-val_p) * len(train_set))
val_size   = len(train_set) - train_size

train_set, val_set = random_split(train_set, [train_size, val_size])

train_loader = DataLoader(dataset=train_set, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(dataset=val_set,   batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(dataset=test_set,  batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)



print("train/val/test:", len(train_set), len(val_set), len(test_set))

Defining Regnet Model

In [ ]:
# Regnet

import torchvision.models as models

model = models.regnet_y_8gf(weights=models.RegNet_Y_8GF_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

# Unfreeze block3 + block4
for param in model.trunk_output.block3.parameters():
    param.requires_grad = True
for param in model.trunk_output.block4.parameters():
    param.requires_grad = True

number_classes = 100
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, number_classes)
)
model = model.to(device)

optimizer = torch.optim.AdamW([
    {"params": model.trunk_output.block3.parameters(), "lr": 5e-5},
    {"params": model.trunk_output.block4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(),                  "lr": 1e-3},
], weight_decay=5e-3)

epochs = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

Training Function

In [ ]:
from torch.amp import autocast, GradScaler

scaler = GradScaler()
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_one_epoch(model, loader):
    """Train for one epoch and return (avg_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            prediction = model(X)
            loss = criterion(prediction, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        predicted = prediction.argmax(dim=1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total

Validation Function

In [ ]:
# Q3.2: Validation function
@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model and return (avg_loss, accuracy)."""

    model.eval()                                 # - Set model to evaluation mode

    total_loss = 0.0                             # - Creating variables for loss and accuracy calculations
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)        # - Loop through batches without computing gradients
        prediction = model(X)
        loss = criterion(prediction,y)
        total_loss += loss.item()
        predicted = prediction.argmax(dim=1)
        correct+= (predicted == y).sum().item()
        total += y.size(0)
    avg_loss = total_loss/len(loader)            # computing loss and accuracy
    accuracy = correct/total

    return avg_loss, accuracy

Training Loop

In [ ]:
import heapq

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = -1
ckpt_dir = "./checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

top3 = []  # min-heap of (val_acc, epoch, path)

for i in range(epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader)
    valid_loss, valid_accuracy = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_accuracy)
    history["val_loss"].append(valid_loss)
    history["val_acc"].append(valid_accuracy)

    # Save every candidate checkpoint
    ckpt_path = f"{ckpt_dir}/epoch_{i+1:02d}_val{valid_accuracy:.4f}.pt"
    torch.save({"model_state_dict": model.state_dict(), "epoch": i}, ckpt_path)

    # Maintain top-3 heap
    heapq.heappush(top3, (valid_accuracy, i, ckpt_path))
    if len(top3) > 3:
        _, _, old_ckpt = heapq.heappop(top3)   # removes lowest of the 4
        if os.path.exists(old_ckpt):
            os.remove(old_ckpt)                 # clean up disk

    # Still track single best for reference
    if valid_accuracy > best_val_acc:
        best_val_acc = valid_accuracy
        best_epoch = i

    print(f"[{i+1:02d}/{epochs}] train_loss: {train_loss:.4f} train_acc: {train_accuracy:.4f} | val_loss: {valid_loss:.4f} val_acc: {valid_accuracy:.4f}")

print(f"\nBest val acc: {best_val_acc:.4f} at epoch {best_epoch+1}")
print("\nTop 3 checkpoints saved:")
for acc, ep, path in sorted(top3, reverse=True):
    print(f"  Epoch {ep+1:02d}: val_acc={acc:.4f} → {path}")

In [ ]:
import pandas as pd

@torch.no_grad()
def ensemble_predict(top3_heap, loader):
    """Average softmax probs across top-3 checkpoints with TTA."""
    top3_sorted = sorted(top3_heap, reverse=True)   # best first
    all_probs = []

    for acc, ep, ckpt_path in top3_sorted:
        print(f"Loading epoch {ep+1} (val_acc={acc:.4f})...")

        # Re-init model architecture
        m = models.regnet_y_8gf(weights=None)
        for param in m.parameters():
            param.requires_grad = False
        for param in m.trunk_output.block3.parameters():
            param.requires_grad = True
        for param in m.trunk_output.block4.parameters():
            param.requires_grad = True
        m.fc = nn.Sequential(nn.Dropout(p=0.0),     # dropout off at inference
                             nn.Linear(in_features, number_classes))
        m.load_state_dict(torch.load(ckpt_path)["model_state_dict"])
        m = m.to(device)
        m.eval()

        # TTA transform
        tta_tf = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.uint8, scale=True),
            v2.Resize((256, 256)),
            v2.RandomResizedCrop(224, scale=(0.7, 1.0), antialias=True),
            v2.RandomHorizontalFlip(p=0.5),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean, std),
        ])

        n_augments = 8
        ckpt_probs = []

        for X, _ in loader:
            X = X.to(device)
            batch_logits = torch.zeros(X.size(0), number_classes).to(device)

            for _ in range(n_augments):
                aug = torch.stack([tta_tf(img.cpu()) for img in X]).to(device)
                with autocast(device_type='cuda'):
                    batch_logits += torch.softmax(m(aug), dim=1)

            batch_logits /= n_augments
            ckpt_probs.append(batch_logits.cpu())

        all_probs.append(torch.cat(ckpt_probs))     # (n_test, 100)
        print(f"  Done.")

    # Average across all 3 checkpoints
    avg_probs = torch.stack(all_probs).mean(0)      # (n_test, 100)
    return avg_probs.argmax(dim=1).numpy()

# Run ensemble inference
print("Running ensemble inference (3 checkpoints × 8 TTA passes)...")
preds = ensemble_predict(top3, test_loader)

# Generate submission CSV
filenames = [test_set.images[i] for i in range(len(test_set))]
ids = [int(f.replace(".jpg", "")) for f in filenames]
submission = pd.DataFrame({"ID": ids, "Label": preds})
submission = submission.sort_values("ID").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv saved — {len(submission)} rows")
print(submission.head(10))

Curves

In [ ]:
# Q3.4: Plot curves
# ========== YOUR CODE STARTS HERE ==========
# TODO:
# - Create two plots: one for loss (train vs val), one for accuracy (train vs val)
# - Use the history dictionary to get the values
# - Add labels, legends, and display the plots

# Loss ------------------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy --------------------------------------------------------------------

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'],   label='Val Acc')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# ========== YOUR CODE ENDS HERE ============

Testing

In [ ]:
# Q4: Test evaluation
# ========== YOUR CODE STARTS HERE ==========
# TODO:
# - Load the best checkpoint (it's a dictionary with 'model_state_dict' and 'epoch')
# - Load the model state from the checkpoint
# - Evaluate on the test set
# - Print test loss and accuracy

checkpoint = torch.load(ckpt_path)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Evaluate the best model state from the checkpoint
test_loss, test_acc = evaluate(model, val_loader)
# ========== YOUR CODE ENDS HERE ============

print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")